# 09 — Tahap B LLM Judge Penuh: v3 → v4

Runner GPU + laporan untuk issue #3. Menjalankan `judge_quality.run()` pada
`easy_clean_v3.jsonl` dan `numglue_clean_v3.jsonl`, lalu melaporkan drop per alasan,
isi keranjang REVIEW, dan perbandingan v3 vs v4.

> ## ⚠️ BELUM DIJALANKAN
>
> Notebook ini ter-commit **tanpa output**. Angka apa pun yang muncul setelah kamu
> menjalankannya adalah milikmu, bukan milik repo.
>
> Dua hal memblokirnya, keduanya dicek keras oleh sel PRA-TERBANG di bawah:
>
> 1. **Gerbang Q1 belum punya angka.** Issue #3 mensyaratkan presisi/recall Q1 dari
>    kalibrasi (issue #2, `notebooks/revisi/08_kalibrasi_judge.ipynb`). Sampai angka itu
>    ada, `Q1_MODE` tidak boleh dipilih — auto-drop tanpa tahu presisi persis keputusan
>    yang gerbang itu ada untuk mencegah.
> 2. **`judge_quality.run()` belum bisa menulis v4.** Suffix keluarannya `_v3` hardcoded
>    (`judge_quality.py:263` dan `:277`), jadi menjalankannya pada `*_v3.jsonl` akan
>    menimpa berkas masukannya sendiri. Harus diperbaiki di `src/` lebih dulu.

**Notebook ini tipis.** Semua logika penyaringan ada di
`src/data_validation/judge_quality.py`; di sini hanya pemanggilan, tabulasi, dan gambar.

⚠️ **Setelah tahap ini data BEKU.** Perubahan baris apa pun sesudahnya memaksa hitung ulang
Tabel IV/V/IX/X/XI/XII/XIII.

## 0. Bootstrap

In [ ]:
import os, sys
p = os.getcwd()
while not os.path.isdir(os.path.join(p, 'src')) and os.path.dirname(p) != p:
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)

import inspect
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

from src.data_validation import judge_quality as jq

print('repo root:', p)
print('judge default vllm:', jq.DEFAULT_VLLM_JUDGE)

## 1. Konfigurasi

`Q1_MODE` sengaja dibiarkan `None`. Isi hanya setelah membaca angka presisi Q1 di thread
issue #2 — nilainya keputusan berbasis bukti, bukan default yang nyaman.

| Nilai | Kapan dipakai | Akibat pada baris Q1=TIDAK |
|---|---|---|
| `'drop'` | presisi vonis Q1 tinggi | dibuang ke `*_v4_dropped.jsonl`, alasan `soal_tak_terjawab` |
| `'review'` | presisi vonis Q1 **rendah** | tidak dibuang; masuk `*_v4_review.jsonl` untuk sapuan manual |

Keranjang REVIEW (Q2=TIDAK, `gold_meragukan`) **tidak pernah** auto-drop, apa pun `Q1_MODE`.

In [ ]:
# Isi setelah angka presisi Q1 tersedia. Jangan menebak.
Q1_MODE = None            # 'drop' | 'review'

# Angka dari issue #2 — dicatat di sini supaya keputusan di atas bisa diaudit belakangan.
Q1_PRESISI = None         # mis. 0.82
Q1_PRESISI_CI = None      # mis. (0.61, 0.93) -- Wilson, jangan titik estimasi sendirian
Q1_RECALL = None
Q1_RECALL_CI = None

DATASET = ['data/Final/easy_clean_v3.jsonl', 'data/Final/numglue_clean_v3.jsonl']
OUT_DIR = Path('data/Final')
VERSI_KELUARAN = 'v4'

JUDGE_BACKEND = 'vllm'
JUDGE_MODEL = jq.DEFAULT_VLLM_JUDGE          # Qwen/Qwen2.5-7B-Instruct
TENSOR_PARALLEL_SIZE = 1                     # A6000 -> 1; Kaggle 2xT4 -> 2
BATCH_SIZE = 64

for f in DATASET:
    assert Path(f).exists(), f'masukan hilang: {f}'

def baris(p) -> int:
    """Hitung baris non-kosong sebuah berkas JSONL."""
    with open(p, encoding='utf-8') as f:
        return sum(1 for b in f if b.strip())

print('masukan siap:')
for f in DATASET:
    print(f'  {f}  {baris(f)} baris')

## 2. PRA-TERBANG

Sel ini gagal keras kalau prasyarat belum terpenuhi. Itu tujuannya. Menjalankan judge dengan
salah satu syarat bolong menghasilkan data beku yang salah, dan biayanya tabel-tabel paper.

In [ ]:
masalah = []

# (a) Gerbang keputusan Q1 -- issue #2 harus sudah memberi angka
if Q1_MODE not in ('drop', 'review'):
    masalah.append(
        'Q1_MODE belum dipilih. Baca presisi/recall Q1 di komentar issue #2 dulu. '
        "Kalau presisi vonis Q1 rendah -> 'review' (JANGAN auto-drop).")
elif Q1_PRESISI is None:
    masalah.append(
        f'Q1_MODE={Q1_MODE!r} dipilih tapi Q1_PRESISI kosong. '
        'Keputusan tanpa angka tidak bisa diaudit; isi angkanya beserta CI Wilson.')

# (b) judge_quality harus bisa menulis versi keluaran yang berbeda dari masukan
sig = inspect.signature(jq.run)
if 'out_version' not in sig.parameters:
    masalah.append(
        "judge_quality.run() belum punya parameter out_version -- suffix '_v3' masih "
        'hardcoded di judge_quality.py:263 dan :277. Menjalankannya pada *_v3.jsonl akan '
        'MENIMPA berkas masukan. Perbaiki di src/ lebih dulu (issue terpisah).')

# (c) tabrakan nama keluaran vs masukan -- cek eksplisit, jangan percaya (b) saja
for f in DATASET:
    stem = jq.VERSION_SUFFIX.sub('', Path(f).stem)
    calon = OUT_DIR / f'{stem}_{VERSI_KELUARAN}.jsonl'
    if calon.resolve() == Path(f).resolve():
        masalah.append(f'keluaran menimpa masukan: {f}')

# (d) Q1_MODE='review' butuh dukungan kode; jalur itu belum ada di run()
if Q1_MODE == 'review' and 'q1_mode' not in sig.parameters:
    masalah.append(
        "Q1_MODE='review' diminta tapi judge_quality.run() belum punya parameter q1_mode. "
        'judge_quality.py:250-251 selalu auto-drop Q1=TIDAK. Jalur review-only belum ada.')

# (e) GPU
try:
    import torch
    if not torch.cuda.is_available():
        masalah.append('CUDA tidak tersedia.')
    else:
        gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
        total = gb * torch.cuda.device_count()
        print(f'GPU: {torch.cuda.get_device_name(0)}  {gb:.1f} GiB x {torch.cuda.device_count()}')
        if total < 15:
            masalah.append(f'VRAM total {total:.1f} GiB < ~15 GiB yang dibutuhkan '
                           f'{JUDGE_MODEL} fp16.')
except ImportError:
    masalah.append('torch tidak terpasang.')

if JUDGE_BACKEND == 'vllm':
    try:
        import vllm
        print('vllm:', vllm.__version__)
    except ImportError:
        masalah.append('vllm tidak terpasang.')

if masalah:
    print('PRA-TERBANG GAGAL\n')
    for m in masalah:
        print('  -', m)
    raise SystemExit('Jangan lanjut. Perbaiki dulu; data setelah tahap ini BEKU.')
print('\nPRA-TERBANG LOLOS')

## 3. Jalankan judge

`run_judge` menulis `.progress` per pertanyaan (`{stem}_q1.progress`, `{stem}_q2.progress`),
jadi eksekusi yang putus bisa dilanjutkan tanpa mengulang vonis yang sudah ada.

Beban terukur dari berkas masukan: 2.254 (`easy_clean_v3`) + 3.762 (`numglue_clean_v3`)
= 6.016 baris, dikurangi yang tersaring tahap A, dikali 2 pertanyaan pada Qwen2.5-7B.
Hitungan jam, bukan menit.

In [ ]:
hasil = {}
for f in DATASET:
    nama = Path(f).stem
    print(f"\n{'=' * 60}\n{nama}\n{'=' * 60}")
    hasil[nama] = jq.run(
        Path(f), OUT_DIR,
        use_llm=True,
        judge_backend=JUDGE_BACKEND,
        judge_model=JUDGE_MODEL,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        batch_size=BATCH_SIZE,
        out_version=VERSI_KELUARAN,
        q1_mode=Q1_MODE,
    )
    for k, v in hasil[nama].items():
        print(f'  {k:14}: {v}')

## 4. Drop per alasan

Tahap A (regex) dan tahap B (judge) dipisah supaya terlihat berapa yang benar-benar ditangkap
judge. Konteks dari issue #2: pada sampel kalibrasi 260 baris, tahap A menangkap **nol** dari
10 positif manual Q1 — seluruh beban Q1 ada di judge.

In [ ]:
def muat_laporan(nama_stem: str) -> dict:
    """Baca *_v4_report.json milik satu dataset."""
    stem = jq.VERSION_SUFFIX.sub('', nama_stem)
    p = OUT_DIR / f'{stem}_{VERSI_KELUARAN}_report.json'
    return json.loads(p.read_text(encoding='utf-8'))


def alasan_per_tahap(nama_stem: str) -> Counter:
    """Pisah alasan drop menurut tahap A (regex) dan tahap B (judge)."""
    stem = jq.VERSION_SUFFIX.sub('', nama_stem)
    p = OUT_DIR / f'{stem}_{VERSI_KELUARAN}_dropped.jsonl'
    c = Counter()
    with open(p, encoding='utf-8') as f:
        for b in f:
            if b.strip():
                r = json.loads(b)
                c[(r.get('_stage', '?'), r.get('_reason', '?'))] += 1
    return c


laporan = {Path(f).stem: muat_laporan(Path(f).stem) for f in DATASET}

for nama, lap in laporan.items():
    print(f'\n{nama}')
    print(f"  masuk {lap['input']:5d} -> keep {lap['keep']:5d} | "
          f"drop {lap['dropped']:4d} | review {lap['review']:4d}")
    for (tahap, alasan), n in sorted(alasan_per_tahap(nama).items()):
        print(f'    [{tahap}] {alasan:22s} {n:5d}')

In [ ]:
# Gambar 1: drop per alasan, tahap A vs tahap B, kedua dataset
fig, axes = plt.subplots(1, len(laporan), figsize=(6 * len(laporan), 4), squeeze=False)
for ax, nama in zip(axes[0], laporan):
    c = alasan_per_tahap(nama)
    label = [f'{a}\n({t})' for (t, a) in c]
    warna = ['#8a8a8a' if t == 'A' else '#d64545' for (t, _a) in c]
    ax.bar(label, list(c.values()), color=warna)
    ax.set_title(f'{nama}: {sum(c.values())} baris dibuang')
    ax.set_ylabel('jumlah baris')
    ax.tick_params(axis='x', rotation=30)
fig.suptitle('Drop per alasan — abu-abu = tahap A (regex), merah = tahap B (judge)')
plt.tight_layout()
plt.show()

## 5. Keranjang REVIEW

Q2=TIDAK (`gold_meragukan`) tidak pernah dibuang otomatis. Isinya dibaca mata satu per satu.
Issue #3 memperkirakan sapuan ini <1 jam.

Kalau `Q1_MODE='review'`, baris `soal_tak_terjawab` ikut mendarat di keranjang yang sama.

In [ ]:
def muat_review(nama_stem: str) -> list[dict]:
    """Baca *_v4_review.jsonl milik satu dataset."""
    stem = jq.VERSION_SUFFIX.sub('', nama_stem)
    p = OUT_DIR / f'{stem}_{VERSI_KELUARAN}_review.jsonl'
    with open(p, encoding='utf-8') as f:
        return [json.loads(b) for b in f if b.strip()]


review = {nama: muat_review(nama) for nama in laporan}
for nama, rs in review.items():
    sebaran = dict(Counter(r.get('_reason', '?') for r in rs))
    print(f'{nama:22s} keranjang review: {len(rs):4d} baris  {sebaran}')

In [ ]:
# Cetak keranjang untuk sapuan manual. Baca semuanya; jangan disampel.
BATAS_TAMPIL = None       # None = semua

for nama, rs in review.items():
    print(f"\n{'=' * 70}\n{nama} — {len(rs)} baris\n{'=' * 70}")
    for i, r in enumerate(rs[:BATAS_TAMPIL]):
        print(f"\n[{i}] alasan={r.get('_reason')}")
        print(f"  soal   : {str(r.get('soal', ''))[:300]}")
        print(f"  jawaban: {str(r.get('jawaban', ''))[:200]}")

### Hasil sapuan manual

Isi `VONIS_MANUAL` sambil membaca sel di atas: indeks baris yang **benar-benar rusak** dan
harus dibuang dari v4. Indeks 0-based terhadap keranjang review dataset tersebut.

Biarkan `SUDAH_DISAPU = False` selama keranjang belum dibaca — sel di bawah menolak jalan,
supaya keranjang yang belum disapu tidak diam-diam ikut lolos ke data latih.

Sel ini **menambah** baris ke `*_v4.jsonl` (mode `'a'`). Menjalankannya dua kali menduplikasi
baris. Kalau perlu mengulang, bangun ulang v4 dari sel bagian 3.

In [ ]:
VONIS_MANUAL = {
    # 'easy_clean_v3':    [],   # <- isi indeks yang harus dibuang
    # 'numglue_clean_v3': [],
}

SUDAH_DISAPU = False     # set True hanya setelah keranjang benar-benar dibaca

if not SUDAH_DISAPU:
    raise SystemExit('Keranjang review belum disapu manual. Acceptance #3 butuh ini.')

for nama, buang in VONIS_MANUAL.items():
    stem = jq.VERSION_SUFFIX.sub('', nama)
    rs = review[nama]
    dibuang = set(buang)
    sisa = [r for i, r in enumerate(rs) if i not in dibuang]
    print(f'{nama}: {len(dibuang)} dibuang manual, {len(sisa)} dinyatakan wajar')

    # baris yang lolos sapuan dikembalikan ke keep
    with open(OUT_DIR / f'{stem}_{VERSI_KELUARAN}.jsonl', 'a', encoding='utf-8') as f:
        for r in sisa:
            bersih = {k: v for k, v in r.items() if not k.startswith('_')}
            f.write(json.dumps(bersih, ensure_ascii=False) + '\n')

    # yang dibuang dicatat, jangan hilang tanpa jejak
    with open(OUT_DIR / f'{stem}_{VERSI_KELUARAN}_dropped.jsonl', 'a', encoding='utf-8') as f:
        for i in dibuang:
            f.write(json.dumps({**rs[i], '_reason': 'gold_rusak_manual', '_stage': 'C'},
                               ensure_ascii=False) + '\n')

## 6. Perbandingan v3 vs v4

Angka inilah yang masuk ke komentar issue #3 dan ke tabel paper.

In [ ]:
print(f"{'dataset':24s} {'v3':>7s} {'v4':>7s} {'delta':>7s} {'%':>7s}")
print('-' * 58)
total_v3 = total_v4 = 0
for f in DATASET:
    stem = jq.VERSION_SUFFIX.sub('', Path(f).stem)
    n3 = baris(f)
    n4 = baris(OUT_DIR / f'{stem}_{VERSI_KELUARAN}.jsonl')
    total_v3 += n3
    total_v4 += n4
    print(f'{stem:24s} {n3:7d} {n4:7d} {n4 - n3:+7d} {(n4 - n3) / n3 * 100:+6.1f}%')
print('-' * 58)
print(f"{'TOTAL':24s} {total_v3:7d} {total_v4:7d} {total_v4 - total_v3:+7d} "
      f'{(total_v4 - total_v3) / total_v3 * 100:+6.1f}%')

In [ ]:
# Gambar 2: v3 vs v4 per dataset
stems = [jq.VERSION_SUFFIX.sub('', Path(f).stem) for f in DATASET]
v3 = [baris(f) for f in DATASET]
v4 = [baris(OUT_DIR / f'{s}_{VERSI_KELUARAN}.jsonl') for s in stems]
x = range(len(stems))

plt.figure(figsize=(7, 4))
plt.bar([i - 0.2 for i in x], v3, width=0.4, label='v3', color='#8a8a8a')
plt.bar([i + 0.2 for i in x], v4, width=0.4, label='v4 (pasca judge)', color='#3b7dd8')
for i, (a, b) in enumerate(zip(v3, v4)):
    plt.text(i + 0.2, b, f'{b - a:+d}', ha='center', va='bottom', fontsize=9)
plt.xticks(list(x), stems)
plt.ylabel('jumlah baris')
plt.title('Ukuran data latih sebelum dan sesudah Tahap B')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Ringkasan untuk komentar issue #3

Salin keluaran sel ini ke thread. Agen #4 membaca angka ini, bukan mengingatnya.

In [ ]:
print('## Hasil Tahap B v3 -> v4\n')
print(f'Judge      : {JUDGE_MODEL} (backend {JUDGE_BACKEND}, tp={TENSOR_PARALLEL_SIZE})')
print(f'Q1_MODE    : {Q1_MODE}  (presisi Q1 {Q1_PRESISI}, CI {Q1_PRESISI_CI})')
print()
for f in DATASET:
    stem = jq.VERSION_SUFFIX.sub('', Path(f).stem)
    lap = laporan[Path(f).stem]
    n4 = baris(OUT_DIR / f'{stem}_{VERSI_KELUARAN}.jsonl')
    print(stem)
    print(f"  v3 {lap['input']} -> v4 {n4}  (drop {lap['dropped']}, review {lap['review']})")
    for (tahap, alasan), n in sorted(alasan_per_tahap(Path(f).stem).items()):
        print(f'    [{tahap}] {alasan:22s} {n}')
print(f'\nTOTAL v3 {total_v3} -> v4 {total_v4}  ({total_v4 - total_v3:+d})')
print('\nDATA SEKARANG BEKU. Perubahan baris sesudah ini memaksa hitung ulang '
      'Tabel IV/V/IX/X/XI/XII/XIII.')